# Qwen3.5-27B + vLLM + OpenAI Client Quickstart

この Notebook は、Docker で vLLM の OpenAI 互換 API サーバを起動したあとに、Python の OpenAI ライブラリから Qwen3.5-27B を使う最短手順をまとめたものです。


## 1. 前提
- Docker が使える
- NVIDIA GPU が使える
- `HF_TOKEN` が必要に応じて設定されている


In [1]:
!docker ps --filter name=qwen35-vllm-27b
!curl -s http://127.0.0.1:8000/v1/models || true


CONTAINER ID   IMAGE                            COMMAND                  CREATED         STATUS         PORTS                                           NAMES
c9b94f45e553   vllm/vllm-openai:cu130-nightly   "vllm serve Qwen/Qwe…"   9 minutes ago   Up 9 minutes   0.0.0.0:30010->8000/tcp, [::]:30010->8000/tcp   qwen35-vllm-27b


## 2. OpenAI クライアント初期化


In [2]:
import os
from openai import OpenAI

BASE_URL = os.environ.get('OPENAI_BASE_URL', 'http://127.0.0.1:8000/v1')
client = OpenAI(api_key='EMPTY', base_url=BASE_URL, timeout=3600)
print('Using base_url =', BASE_URL)
client


Using base_url = http://127.0.0.1:30010/v1


## 3. テキスト推論


In [3]:
resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[{'role': 'user', 'content': 'vLLMとは何かを日本語で2文で説明してください。'}],
    max_tokens=64,
)
resp


ChatCompletion(id='chatcmpl-8ede8aac98d20b3d', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: vLLM (a library for serving large language models).\n    *   Language: Japanese.\n    *   Constraint: Exactly 2 sentences (2 文).\n\n2.  **Identify Key Information about'), stop_reason=None, token_ids=None)], created=1774940897, model='Qwen/Qwen3.5-27B', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=64, prompt_tokens=24, total_tokens=88, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None, prompt_token_ids=None, kv_transfer_params=None)

## 4. 画像入力推論


In [4]:
import base64
import mimetypes
from pathlib import Path

img = Path('../assets/sample_shapes.png')
mime = mimetypes.guess_type(img.name)[0] or 'image/png'
image_url = 'data:' + mime + ';base64,' + base64.b64encode(img.read_bytes()).decode('utf-8')

resp = client.chat.completions.create(
    model='Qwen/Qwen3.5-27B',
    messages=[
        {'role': 'user', 'content': [
            {'type': 'text', 'text': 'この画像の図形の数と色を説明してください。'},
            {'type': 'image_url', 'image_url': {'url': image_url}},
        ]}
    ],
    max_tokens=64,
)
resp


ChatCompletion(id='chatcmpl-91d300453c507b1d', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='ユーザーは提供された画像にある図形の数と色について説明を求めています。\n\n1.  **画像の分析:**\n    *   タイトル: "Sample Shapes"\n    *   左側: 青い四角形（正方形に近い長方形）。ラベルには "blue rectangle'), stop_reason=None, token_ids=None)], created=1774940912, model='Qwen/Qwen3.5-27B', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=64, prompt_tokens=625, total_tokens=689, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None, prompt_token_ids=None, kv_transfer_params=None)